# Global Air Quality Analysis Using PySpark

### Project Overview

This project uses PySpark to process, clean, and analyze global air quality data by combining WHO pollutant measurements (PM2.5, PM10, NO2) with country-level population data. The main goal is to calculate meaningful metrics such as a Pollution Index and a population-weighted Health Risk Index, enabling cross-country and temporal comparisons of air pollution and its impact on human health. The analysis focuses on identifying countries with the highest pollution exposure and population health risks, providing actionable insights for policymakers.

### Workflow Structure

The notebook demonstrates a complete PySpark data pipeline, starting from raw data ingestion to advanced cleaning, merging datasets, and deriving composite indices. Key steps include standardizing country names, handling missing or extreme values, imputing pollutant and population data, and calculating the Pollution Index and Health Risk Index. By combining pollutant intensity with population exposure, the project highlights countries like Pakistan, Bangladesh, and India as high-risk areas. The workflow also includes visualizations such as bubble charts, line charts, and stacked bar charts to communicate patterns and trends effectively. This approach showcases scalable data processing with Spark while providing clear, project-specific insights into global air pollution and health risks.

******************************

## Choice of Technology: Why Apache Spark

For this project, we chose **Apache Spark** to build and run our data pipeline. The decision was based on several practical and technical reasons:

### 1. Scalability and Distributed Processing
- Our dataset combines global air pollution data with population figures, giving us **tens of thousands of rows**.  
- Apache Spark can process data **across multiple machines**, making it fast and efficient.  
- Unlike tools like Pandas, Spark can **handle much larger datasets** without slowing down, which is useful for showing cloud-scale data processing.

### 2. Powerful DataFrame API and SQL
- Spark’s **DataFrame API** makes it easy to do complex tasks like:  
  - Filling or dropping missing values (`fill`, `dropna`, `coalesce`)  
  - Aggregations (`groupBy`, `agg`, `percentile_approx`)  
  - Conditional transformations (`withColumn`, `when`, `lit`)  
- We could also use **Spark SQL** for queries if needed, giving extra flexibility.

### 3. Performance Benefits
- Spark uses **lazy evaluation** and **query optimization**, so multiple steps run efficiently together.  
- It **parallelizes computations**, which reduces runtime compared to doing everything step by step in Python.

## 4. Why Not Other Technologies

We also considered other big data tools, but they were not the best fit for this project

- **Apache MapReduce** can process large datasets, but it is low-level and requires writing a lot of boilerplate code for each step. Implementing cleaning, transformations, and aggregations would be slow and cumbersome.

- **Hive** is useful for running SQL queries on structured data, but it is harder to perform step-by-step transformations, handle missing values, or calculate custom metrics like Pollution Index or Health Risk Index.

- **Storm** is designed for real-time stream processing, but our dataset is static. Using Storm would add unnecessary complexity without any benefit.

- **Apache Spark**, in contrast, combines high-level APIs with distributed processing. It supports batch operations, in-memory computation, SQL queries, and complex transformations efficiently. It also integrates well with Python for visualization, making it ideal for both processing and analyzing large datasets in a single workflow.

### 5. Easy Integration and Visualization
- Spark works well with **Python (PySpark)**, letting us run all steps in a **Jupyter notebook**.  
- This also makes it simple to visualize results and create reports, while heavy transformations are still done efficiently in Spark.

### 6. Conclusion
Using Apache Spark allows us to:  
- Process large datasets efficiently  
- Clean and transform data step by step  
- Calculate meaningful metrics like `pollution_index` and `health_risk_index`  
- Ensure scalability, speed, and reproducibility

*******************************************

# Step 1: Initialize Spark and Load Data

In [ ]:
from pyspark.sql import SparkSession

if SparkSession._instantiatedSession is not None:
    SparkSession._instantiatedSession.stop()

spark = SparkSession.builder \
    .appName("WHO-Population Integration") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

print("Spark session started")
print("App Name:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)

In [ ]:
import os
import pandas as pd
import tempfile

who_xlsx_path = os.path.join("Datasets", "WHO Dataset.xlsx")
pop_csv_path  = os.path.join("Datasets", "POPULATION Dataset.csv")

# data is on the second sheet, not the default README sheet
who_pd = pd.read_excel(who_xlsx_path, sheet_name="AAP_2022_city_v9", engine="openpyxl")
_tmp = os.path.join(tempfile.gettempdir(), "who_tmp.csv")
who_pd.to_csv(_tmp, index=False, encoding="utf-8")
df_who = spark.read.csv(_tmp, header=True, inferSchema=True)

df_pop = spark.read.csv(pop_csv_path, header=True, inferSchema=True)

print("WHO Dataset Schema:")
df_who.printSchema()
df_who.show(5, truncate=False)

print("\nPopulation Dataset Schema:")
df_pop.printSchema()
df_pop.show(5, truncate=False)

### Step 1: Setting Up Spark and Loading Data

**Goal:**  
Here we start by setting up Spark and loading the two main datasets we will use: the WHO air quality data and the population data. This is the first step to make sure we can handle large datasets and perform all the necessary analysis.  

**What we did:**  
- **Start Spark:** Created a SparkSession called WHO-Population Integration to work with Spark DataFrames.  
- **Load files:** Read the CSV files using spark.read.csv with headers and automatic detection of column types.  
- **Check the data:** Looked at the schema and a few sample rows with printSchema and show to understand what the data looks like.  

**WHO Air Quality Data:**  
- **Size and columns:** 32,196 rows with columns like region, country, city, year, PM2.5, PM10, NO2, coverage percentages, reference, monitoring stations, version, and status.  
- **Notes:** Some pollutant values are missing for certain city-year pairs. This is normal and we will handle it later.  

**Population Data:**  
- **Size and columns:** 16,930 rows with country name, code, year, and population value.  
- **Notes:** Each row is for a country in a given year, covering multiple years.  

**Why we did this:**  
- Loading both datasets at the start lets us combine them later using country and year.  
- Checking the schema and sample rows makes sure Spark understood the data correctly and shows if there are any immediate issues.  
- This step sets the stage for cleaning, transforming, and merging the data in the next steps.

***********************************

# Step 2: Population Dataset Cleaning

In [ ]:
from pyspark.sql.functions import col, lower, trim, count, when, lit

for old, new in {"Country Name": "country_name", "Country Code": "country_code",
                 "Year": "year", "Value": "population"}.items():
    df_pop = df_pop.withColumnRenamed(old, new)

for c in ["country_name", "country_code"]:
    df_pop = df_pop.withColumn(c, lower(trim(col(c))))

df_pop.select([
    (count(when(col(c).isNull(), c)) / count(lit(1)) * 100).alias(c + "_missing_pct")
    for c in df_pop.columns
]).show(truncate=False)

df_pop.select(["year", "population"]).describe().show()
df_pop.select("country_name").distinct().show()
df_pop.select("year").distinct().show()

df_pop.printSchema()
df_pop.show(5, truncate=False)

### Step 2: Cleaning the Population Data

**Goal:**  
We cleaned and standardized the population dataset so it can be easily combined with the WHO air quality data later.  

**What we did:**  
- **Rename columns:** Changed all column names to lowercase: country_name, country_code, year, population. This makes it easier to work with them in Spark.  
- **Standardize strings:** Converted country names and codes to lowercase and removed extra spaces to avoid mismatches with WHO data.  
- **Check missing data:** Calculated how many values are missing in each column.  
- **Get basic stats:** Checked numbers for year and population to see the ranges and averages.  
- **Check distinct values:** Looked at the number of unique countries and years to understand coverage.  

**What we found:**  
- **Missing values:** None, all columns are complete.  
- **Numbers:**  
  - Years go from 1960 to 2023, with an average around 1991.  
  - Population ranges from 2,715 to over 8 billion, with an average around 216 million.  
- **Distinct countries and years:** Covers many countries over many years, so we can match with the WHO dataset.  
- **Sample rows:** Top rows show lowercase country names and codes, ready for merging.  

**Why this matters:**  
Cleaning and standardizing the population data prevents join errors with the WHO data and ensures accurate results in later analysis.

**************************